# Pipeline de tri de pages financières (PDF normaux + scannés)

Ce notebook orchestre un pipeline **modulaire** composé de fichiers Python séparés :

| Fichier | Rôle |
|---|---|
| `config.py` | Mots-clés EN/FR, poids, seuils — **modifiez les règles ici** |
| `text_extraction.py` | Extraction de texte (natif / OCR / hybride) — **remplacez l'OCR ici** |
| `relevance.py` | Scoring de pertinence — calcule **2 méthodes en parallèle** (voir ci-dessous) |
| `pdf_processor.py` | Traite un PDF ou un dossier entier, écrit un PDF filtré par méthode |
| `report.py` | Génère le fichier Excel global, avec les 2 méthodes côte à côte |
| `density_tester.py` | Outil de calibrage : densité + résultat des 2 méthodes, page par page |

## Deux méthodes de décision, calculées en parallèle

Chaque page est évaluée selon **2 méthodes indépendantes**, pour pouvoir les comparer :

1. **`density_only`** (densité seule) : la page est pertinente si sa densité
   numérique dépasse le seuil, **que le mot-clé matche ou non**.
2. **`keyword_and_density`** (mot-clé + densité) : la page est pertinente
   **seulement si** un mot-clé matche **ET** que la densité numérique confirme.

`settings["methods"]` contrôle lesquelles sont utilisées pour produire un
PDF filtré :
- **Les deux** (par défaut) → deux PDF filtrés par document, dans
  `output_pdfs/density_only/` et `output_pdfs/keyword_and_density/`, plus
  une colonne "Pages où les méthodes diffèrent" dans le rapport Excel pour
  repérer rapidement les cas de désaccord.
- **Une seule** (ex: `settings["methods"] = ["keyword_and_density"]`) → un
  seul PDF filtré, écrit directement dans `output_pdfs/`.

**Pourquoi c'est modulaire ?** Chaque brique dépend d'une interface abstraite
(`TextExtractor`, `RelevanceScorer`) et non d'une implémentation précise.
Par exemple, si demain vous voulez remplacer Tesseract par EasyOCR ou une
API cloud, il suffit d'écrire une nouvelle classe héritant de
`TextExtractor` dans `text_extraction.py` — **aucune autre partie du
pipeline n'a besoin de changer**.


## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())  # s'assure que les modules du dossier sont importables

from text_extraction import build_extractor, NativeTextExtractor, TesseractOCRExtractor, HybridTextExtractor
from relevance import build_default_composite_scorer, KeywordScorer, NumericDensityScorer, CompositeScorer
from pdf_processor import process_folder
from report import write_excel_report
import config


## 2. Paramètres

- `INPUT_DIR` : dossier contenant les PDF financiers à traiter (normaux ou scannés)
- `OUTPUT_DIR` : dossier où seront écrits les PDF filtrés (mêmes noms de fichiers, mais seulement les pages pertinentes)
- `EXCEL_REPORT_PATH` : chemin du fichier Excel récapitulatif global (situé **en dehors** de `OUTPUT_DIR`)


In [ ]:
INPUT_DIR = "input_pdfs"                 # <-- dossier avec vos PDF financiers
OUTPUT_DIR = "output_pdfs"               # <-- dossier de sortie (PDF filtrés)
EXCEL_REPORT_PATH = "rapport_global.xlsx"  # <-- fichier Excel global (hors OUTPUT_DIR)

os.makedirs(INPUT_DIR, exist_ok=True)
print(f"Déposez vos PDF dans le dossier: {os.path.abspath(INPUT_DIR)}")


## 3. Configuration des règles de détection (optionnel)

Toutes les valeurs ci-dessous viennent de `config.py`. Vous pouvez soit éditer
directement `config.py`, soit surcharger certains paramètres ici sans toucher
au fichier (utile pour tester rapidement différents seuils).

In [ ]:
settings = dict(config.DEFAULT_SETTINGS)  # copie modifiable

# Exemples de réglages que vous pouvez ajuster :
# settings["methods"] = ["keyword_and_density"]  # une seule méthode (au lieu des 2)
# settings["methods"] = ["density_only"]          # l'autre méthode seule
# settings["extraction_mode"] = "ocr_only"        # force l'OCR sur toutes les pages
# settings["confirmation_threshold"] = 0.30        # confirmation plus permissive
# settings["numeric_density_threshold"] = 0.05
# settings["enable_continuation_detection"] = False
# settings["use_zero_shot"] = True                 # nécessite: pip install transformers torch

# --- exclusion automatique (ex: pages "consolidated"/"consolidé") ---
# settings["exclude_pages_with_excluded_words"] = False   # désactive l'exclusion
# settings["excluded_words"] = ["consolidated", "consolide", "draft"]  # personnalise la liste

# --- gros volumes (beaucoup de PDF / documents longs) ---
# import os
# settings["parallel_workers"] = os.cpu_count() - 1
# settings["enable_checkpoint"] = True
# settings["retry_errors"] = True

settings


## 4. Construction des modules (extracteur de texte + scorer de pertinence)

C'est ICI que vous changeriez de moteur OCR ou de stratégie d'extraction si
besoin. `build_extractor(settings)` choisit automatiquement l'implémentation
selon `settings["extraction_mode"]` :
- `"hybrid"` (par défaut) : texte natif, bascule sur l'OCR si trop court ou
  si l'encodage semble corrompu.
- `"ocr_only"` : force l'OCR sur TOUTES les pages, natif ou pas. À utiliser
  si le mode hybride lit mal vos documents (problème d'encodage de police).
- `"native_only"` : jamais d'OCR, uniquement le texte déjà encodé dans le PDF.

Pour remplacer complètement le moteur OCR (Tesseract -> autre chose), créez
une nouvelle classe héritant de `TextExtractor` dans `text_extraction.py`,
puis adaptez `build_extractor()` pour la retourner. Le reste du pipeline ne
change pas.

In [ ]:
# --- Extraction de texte : choisie automatiquement selon settings["extraction_mode"] ---
extractor = build_extractor(settings)

# --- Scoring de pertinence (mots-clés + densité numérique + zero-shot optionnel) ---
scorer = build_default_composite_scorer(settings)

print("Mode d'extraction:", settings["extraction_mode"])
print("Extracteur:", extractor)
print("Scorer:", scorer)

print("Méthodes actives:", settings["methods"])


## 5. Exécution du pipeline sur le dossier

In [ ]:
results = process_folder(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    extractor=extractor,
    scorer=scorer,
    settings=settings,
    verbose=True,
)


## 6. Génération du rapport Excel global

In [ ]:
write_excel_report(results, EXCEL_REPORT_PATH)
print(f"PDF filtrés dans: {os.path.abspath(OUTPUT_DIR)}")
print(f"Rapport Excel: {os.path.abspath(EXCEL_REPORT_PATH)}")


## 7. Aperçu rapide des résultats dans le notebook

In [ ]:
from report import build_summary_dataframe, build_detail_dataframe

summary_df = build_summary_dataframe(results)
summary_df


In [ ]:
detail_df = build_detail_dataframe(results)
detail_df


## 8. (Optionnel) Tester la détection sur un seul texte

Pratique pour ajuster vos mots-clés dans `config.py` sans relancer tout le dossier.

In [ ]:
sample_text = """
Consolidated Statement of Financial Position
Total assets 1 234 567   Total liabilities 654 321
"""

evaluation = scorer.evaluate(sample_text)
print("Pertinente (densité seule) ?", evaluation.method_results["density_only"])
print("Pertinente (mot-clé + densité) ?", evaluation.method_results["keyword_and_density"])
print("Score:", evaluation.score)
print("Mots-clés trouvés:", evaluation.matched_keywords)
print("Détail:", evaluation.details)


---
### Comment la pertinence est décidée (2 méthodes comparables)

Pour chaque page, dans l'ordre :

1. On cherche un mot-clé de `config.py` (ex: "bilan", "balance sheet"...).
2. On calcule la densité numérique de la page (proportion de chiffres +
   volume absolu de chiffres), **dans tous les cas**.
3. **Deux décisions sont calculées en parallèle** :
   - **`density_only`** : pertinente si la densité dépasse le seuil de
     confirmation, que le mot-clé ait matché ou non.
   - **`keyword_and_density`** : pertinente seulement si un mot-clé a
     matché ET que la densité confirme.

Ces deux méthodes divergent typiquement sur les pages sans mot-clé mais
denses en chiffres (ex: une page qui continue un tableau financier sans
répéter le titre, OU un tableau statistique non-financier). `density_only`
les garde, `keyword_and_density` les rejette (sauf si la détection de
continuation les rattrape - voir plus bas).

`settings["methods"]` contrôle ce qui est produit :
- **`["density_only", "keyword_and_density"]`** (défaut) → deux PDF
  filtrés par document (`output_pdfs/density_only/`,
  `output_pdfs/keyword_and_density/`) + colonnes côte à côte et colonne
  "Pages où les méthodes diffèrent" / "Méthodes en désaccord ?" dans le
  rapport Excel, pour repérer rapidement les cas de divergence à trancher.
- **Une seule méthode** → un seul PDF filtré, écrit directement dans
  `output_pdfs/`.

**Détection de continuation** (filet de sécurité, activé par défaut) : une
page sans mot-clé qui suit immédiatement une page pertinente est acceptée
si elle est assez dense en chiffres (`continuation_threshold`). **Par
défaut, ceci ne s'applique qu'à `density_only`** — `keyword_and_density`
reste une méthode STRICTE où chaque page doit satisfaire le mot-clé ET la
densité elle-même, sans exception de continuation. Réglable via
`settings["continuation_methods"]` (liste des méthodes concernées) si vous
voulez aussi assouplir `keyword_and_density`.

### Exclusion automatique ("consolidated"/"consolidé")

Avant toute autre règle, une page contenant un mot de `EXCLUDED_WORDS`
(`config.py`, par défaut "consolidated"/"consolidé") est **rejetée
d'office pour les DEUX méthodes**, quels que soient les mots-clés trouvés
ou la densité numérique. La continuation ne peut pas non plus rattraper
une page exclue. Réglable via `settings["exclude_pages_with_excluded_words"]`
(désactiver) et `settings["excluded_words"]` (personnaliser la liste).

### Notes sur la modularité

- **Choisir une ou les deux méthodes** : `settings["methods"]`, voir ci-dessus.
- **Changer d'OCR ou de stratégie d'extraction** : `settings["extraction_mode"]`
  accepte `"hybrid"` (par défaut), `"ocr_only"`, ou `"native_only"`. Pour
  changer complètement de moteur OCR, créez une nouvelle classe héritant de
  `TextExtractor` dans `text_extraction.py` et adaptez `build_extractor()`.
- **Changer/ajouter des mots-clés** : éditez uniquement `config.py`
  (`KEYWORDS_EN`, `KEYWORDS_FR`).
- **Activer le zero-shot** : `pip install transformers torch`, puis
  `settings["use_zero_shot"] = True`.
- **Ajuster l'exigence de densité numérique** : `numeric_density_threshold`
  / `numeric_min_digit_count` / `confirmation_threshold` dans `config.py`.
  Utilisez `density_tester.py` (section 9) pour observer l'effet sur vos
  documents et comparer les 2 méthodes avant de relancer tout le pipeline.
- **Ajuster la détection de continuation** : `continuation_threshold`, ou
  `enable_continuation_detection = False` pour la désactiver.

### Gros volumes : reprise après plantage et traitement en parallèle

- **Reprise automatique** : `enable_checkpoint = True` (par défaut) écrit
  une sauvegarde (`<output_dir>/.pipeline_checkpoint.json`) après chaque
  PDF traité. Relancer la même cellule reprend où on s'était arrêté.
- **Traitement en parallèle** : `settings["parallel_workers"] = N`.
  Restez en séquentiel (`N=1`) tant que vous validez vos réglages.
  Incompatible avec `use_zero_shot=True`.
- **Fichiers en erreur** : `retry_errors = True` (par défaut) les retente
  automatiquement au run suivant.


## 9. (Optionnel) Tester la densité numérique d'un PDF page par page

`density_tester.py` réutilise EXACTEMENT le même code d'extraction que le
pipeline (`build_extractor`), et affiche pour chaque page : la méthode
d'extraction utilisée, le nombre de chiffres, la densité (ratio), le score
normalisé, et si elle dépasse le seuil de confirmation actuel.

Pratique pour calibrer `numeric_density_threshold`, `numeric_min_digit_count`
et `confirmation_threshold` sur vos propres documents, sans avoir à relancer
tout le pipeline de filtrage à chaque essai. Voir aussi la section
"Tests unitaires" du module dans `modules.ipynb` pour des tests plus poussés.

In [ ]:
from density_tester import analyze_pdf_density
from IPython.display import display

# Remplacez par le chemin d'un de vos PDF (ou un fichier de input_pdfs/)
test_pdf_path = os.path.join(INPUT_DIR, os.listdir(INPUT_DIR)[0]) if os.listdir(INPUT_DIR) else None

if test_pdf_path:
    density_df = analyze_pdf_density(test_pdf_path, settings)
    print(f"Analyse de: {test_pdf_path}")
    display(density_df)
else:
    print(f"Aucun PDF trouvé dans {INPUT_DIR}")
